In [1]:
import tensorflow as tf
import zipfile,os,shutil
import numpy as np
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if(logs.get('accuracy') > 0.975):
      print("\nTraining stop")
      self.model.stop_training = True

callbacks = myCallback()

In [3]:
base_dir = 'rockpaperscissors'
roc_dir = os.path.join(base_dir,'rock')
pap_dir = os.path.join(base_dir, 'paper')
sci_dir = os.path.join(base_dir, 'scissors')

In [4]:
train_dir = os.path.join(base_dir,'train')
validation_dir = os.path.join(base_dir,'validation')
# os.mkdir(train_dir)
# os.mkdir(validation_dir)

In [5]:

subdirectories = ['train', 'validation']
categories = ['rock', 'paper', 'scissors']

for subdir in subdirectories:
    for category in categories:
        directory = os.path.join(base_dir, subdir, category)
        os.makedirs(directory, exist_ok=True)


In [6]:
train_roc = os.path.join(train_dir, 'rock')
train_pap = os.path.join(train_dir, 'paper')
train_sci = os.path.join(train_dir, 'scissors')
val_roc = os.path.join(validation_dir, 'rock')
val_pap = os.path.join(validation_dir, 'paper')
val_sci = os.path.join(validation_dir, 'scissors')

In [7]:
train_roc_dir, val_roc_dir = train_test_split(os.listdir(roc_dir), test_size = 0.3)
train_pap_dir, val_pap_dir = train_test_split(os.listdir(pap_dir), test_size = 0.3)
train_sci_dir, val_sci_dir = train_test_split(os.listdir(sci_dir), test_size = 0.3)

In [12]:
for gambar in train_roc_dir:
  shutil.copy(os.path.join(roc_dir, gambar), os.path.join(train_roc, gambar))

for gambar in train_pap_dir:
  shutil.copy(os.path.join(pap_dir, gambar), os.path.join(train_pap, gambar))

for gambar in train_sci_dir:
  shutil.copy(os.path.join(sci_dir, gambar), os.path.join(train_sci, gambar))



In [13]:
for gambar in val_roc_dir:
  shutil.copy(os.path.join(roc_dir, gambar), os.path.join(val_roc, gambar))

for gambar in val_pap_dir:
  shutil.copy(os.path.join(pap_dir, gambar), os.path.join(val_pap, gambar))

for gambar in val_sci_dir:
  shutil.copy(os.path.join(sci_dir, gambar), os.path.join(val_sci, gambar))

In [8]:
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    horizontal_flip = True,
    shear_range = 0.2,
    fill_mode = 'nearest',
)
test_datagen = ImageDataGenerator(
    rescale = 1./225,
)

In [9]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(380,380),
    batch_size= 32,
    class_mode='categorical'
)

validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size = (380,380),
    batch_size = 32,
    class_mode = 'categorical'
)

Found 2954 images belonging to 3 classes.
Found 1283 images belonging to 3 classes.


In [10]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3,3), activation = 'relu', input_shape= (380,380,3)),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Conv2D(64,(3,3), activation= 'relu'),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Conv2D(128,(3,3), activation= 'relu'),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Conv2D(512,(3,3), activation= 'relu'),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(512, activation= 'relu'),
  tf.keras.layers.Dense(3, activation= 'softmax')
])

C:\Users\booma\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 378, 378, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 189, 189, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 187, 187, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 93, 93, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 91, 91, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 45, 45, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 43, 43, 512)    │       590,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 21, 21, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 21, 21, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 225792)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │   115,606,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,291,139 (443.62 MB)

 Trainable params: 116,291,139 (443.62 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(loss='categorical_crossentropy',
              optimizer=tf.optimizers.Adam(),
              metrics=['accuracy'])

In [13]:
history = model.fit(
      train_generator,
      steps_per_epoch=20,
      epochs=2,
      validation_data=validation_generator,
      validation_steps=25,
      verbose=2,
        callbacks = [callbacks]
    )

Epoch 1/2


C:\Users\booma\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


In [22]:
import cv2 as cv
def main():
    # Inisialisasi objek capture dari kamera
    cap = cv.VideoCapture(0)

    # Periksa apakah kamera berhasil dibuka
    if not cap.isOpened():
        print("Tidak dapat membuka kamera.")
        return

    # Set ukuran frame yang diinginkan
    frame_width = 350
    frame_height = 350

    # Set ukuran frame
    cap.set(cv.CAP_PROP_FRAME_WIDTH, frame_width)
    cap.set(cv.CAP_PROP_FRAME_HEIGHT, frame_height)

    while True:
        # Tangkap frame dari kamera
        ret, frame = cap.read()

        if not ret:
            print("Gagal mendapatkan frame.")
            break

        # Tampilkan frame dengan ukuran 300x300
        cv.imshow('Live Object', frame)

        # Jika tombol 'q' ditekan, keluar dari loop
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Tutup kamera dan jendela OpenCV
    cap.release()
    cv.destroyAllWindows()

if __name__ == "__main__":
    main()


In [26]:
cap = cv.VideoCapture(0)
prev_x, prev_y, prev_w, prev_h = None, None, None, None

while True:
    ret, frame = cap.read()

    if not ret:
        break

    resized_frame = cv.resize(frame, (380, 380))
    resized_frame = resized_frame.astype('float32') / 255.0

    predictions = model.predict(tf.expand_dims(resized_frame, axis=0))
    predicted_class = tf.argmax(predictions, axis=1).numpy()[0]
    x, y, w, h = 25, 25, 50, 50  
    if prev_x is not None:
        x += (x - prev_x)
        y += (y - prev_y)
    prev_x, prev_y, prev_w, prev_h = x, y, w, h
    

    if str(predicted_class) == "0":
        objek = 'paper'
    elif str(predicted_class) == '1':
         objek = 'rock'
    elif str(predicted_class) == "2":
        objek = 'scissors'
    else:
        objek = 'nothing'
    cv.putText(frame, objek, (x + 60, y + 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv.imshow('Object Detection', frame)

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━

In [24]:
train_generator.class_indices

{'paper': 0, 'rock': 1, 'scissors': 2}

In [25]:
str(predicted_class)
if str(predicted_class) == "0":
     objek = 'kertas'
elif str(predicted_class) == '1':
     objek = 'baru'
else:
     objek = 'gunting'
